# (성공) Tensorflow + HuggingFace

In [1]:
!pip install -q tensorflow transformers pandas scikit-learn

In [2]:
import pandas as pd

import tensorflow as tf
from transformers import BertTokenizer, TFBertForSequenceClassification

# from tensorflow.keras.models import Model
# from tensorflow.keras.layers import Dense, Input
# from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.losses import SparseCategoricalCrossentropy

In [3]:
# 2. 데이터 불러오기
df = pd.read_csv("clean_train_without_tag.csv", header=1, names=["id", "text", "label"])

# text와 label만 추출
df = df[["text", "label"]]

# train / validation split
from sklearn.model_selection import train_test_split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.1,
    random_state=42
)

In [4]:
# 3. 토크나이저 불러오기 (BERT 기본 모델)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", use_fast=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [5]:
# 4. 텍스트를 토큰화
def encode_texts(texts, tokenizer, max_len=128):
    return tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=max_len,
        return_tensors="tf"
    )

train_texts = [str(t) for t in train_texts]
val_texts = [str(v) for v in val_texts]
train_encodings = encode_texts(train_texts, tokenizer)
val_encodings = encode_texts(val_texts, tokenizer)

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


In [6]:
# 6. 모델 불러오기 (3-class 분류)
model = TFBertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3, from_pt=True, force_download=True)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
# 7. 컴파일
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metric = tf.keras.metrics.SparseCategoricalAccuracy("accuracy")

model.compile(optimizer=optimizer, loss=loss, metrics=[metric])

In [8]:
train_dataset = tf.data.Dataset.from_tensor_slices((
    {
        "input_ids": train_encodings["input_ids"],
        "attention_mask": train_encodings["attention_mask"]
    },
    tf.convert_to_tensor(train_labels)
)).shuffle(len(train_labels)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices((
    {
        "input_ids": val_encodings["input_ids"],
        "attention_mask": val_encodings["attention_mask"]
    },
    tf.convert_to_tensor(val_labels)
)).batch(16)

In [9]:
train_labels = [int(t) for t in train_labels]
val_labels = [int(v) for v in val_labels]

In [10]:
# 8. 학습
model.fit(train_dataset, validation_data=val_dataset, epochs=3)

Epoch 1/3
1800/1800 [==============================] - 717s 366ms/step - loss: 0.5045 - accuracy: 0.8049 - val_loss: 0.4218 - val_accuracy: 0.8419
Epoch 2/3
1800/1800 [==============================] - 649s 360ms/step - loss: 0.3389 - accuracy: 0.8751 - val_loss: 0.4302 - val_accuracy: 0.8419
Epoch 3/3
1800/1800 [==============================] - 647s 360ms/step - loss: 0.2152 - accuracy: 0.9225 - val_loss: 0.4682 - val_accuracy: 0.8388


In [11]:
# 9. 예측 함수
def predict(texts):
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=128, return_tensors="tf")
    outputs = model(encodings)
    logits = outputs.logits
    probs = tf.nn.softmax(logits, axis=-1)
    preds = tf.argmax(probs, axis=-1)
    return preds.numpy().tolist()

In [12]:
# 테스트
sample_texts = [
    "I love this game!",
    "This is terrible...",
    "It’s okay, not bad."
]

print(predict(sample_texts))  # → [1, 2, 0] 예상

[1, 2, 2]


In [13]:
# 1. 저장 경로 지정 (예: 'saved_model')
save_path = "./saved_model"

# 2. 모델 저장
model.save_pretrained(save_path)

# 3. 토크나이저 저장
tokenizer.save_pretrained(save_path)

('./saved_model/tokenizer_config.json',
 './saved_model/special_tokens_map.json',
 './saved_model/vocab.txt',
 './saved_model/added_tokens.json')

In [14]:
import shutil
shutil.make_archive('bert_trained_model', 'zip', './saved_model')


'/content/bert_trained_model.zip'

In [15]:
from google.colab import files
files.download('bert_trained_model.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
print(predict("I'm so happy"))

[1]


# (실패, Archive) PyTorch + HuggingFace

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, Adam
from transformers import Trainer, TrainingArguments

# 1. 데이터 불러오기
df = pd.read_csv("clean_train_without_tag.csv", header=None, names=["id", "text", "label"])

# text와 label만 추출
df = df[["text", "label"]]

# train/validation 분리
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.1,
    random_state=42
)

In [ ]:
# 3. Dataset 클래스 정의
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# 2. 토크나이저 불러오기 (기본 BERT)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
# 4. Dataset/Dataloader 생성
train_dataset = TweetDataset(train_texts, train_labels, tokenizer)
val_dataset = TweetDataset(val_texts, val_labels, tokenizer)

# 5. 모델 불러오기 (클래스 3개)
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)

# 6. TrainingArguments & Trainer 설정
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer
)

In [ ]:
# 7. 학습 실행
trainer.train()

In [ ]:
# 8. 예측 함수
def predict(texts):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        preds = torch.argmax(probs, dim=1)
    return preds.tolist()

In [ ]:
# 테스트
sample_texts = [
    "I love this game!",
    "This is terrible...",
    "It’s okay, not bad."
]
print(predict(sample_texts))  # → [1, 2, 0] 같은 결과 예상